In [ ]:
# Build output DataFrame — one row per series per month
records = []

for actual, forecast, original_series in zip(val_inv, pred_series_inv, val_list):

    # Get the series label from unscaled val_list (retains original string value)
    series_name = original_series.static_covariates['PARENT_DEALER_CODE_MODEL_FAMILY'].values[0]

    months          = forecast.time_index
    actual_values   = actual.values().flatten()
    forecast_values = forecast.values().flatten()

    for month, act, pred in zip(months, actual_values, forecast_values):
        records.append({
            'MONTH_OF_SALE'                  : month,
            'PARENT_DEALER_CODE_MODEL_FAMILY' : series_name,
            'ACTUAL_SALES'                   : round(float(act),  2),
            'PREDICTED_SALES'                : round(float(pred), 2)
        })

df_output = pd.DataFrame(records)
df_output['MONTH_OF_SALE'] = pd.to_datetime(df_output['MONTH_OF_SALE']).dt.strftime('%Y-%m-%d')
df_output = df_output.sort_values(['PARENT_DEALER_CODE_MODEL_FAMILY', 'MONTH_OF_SALE']).reset_index(drop=True)

print(f'Output shape : {df_output.shape}')
print(f'Months       : {df_output["MONTH_OF_SALE"].unique()}')
print(f'Series count : {df_output["PARENT_DEALER_CODE_MODEL_FAMILY"].nunique()}')
df_output.head(10)

In [ ]:
['DEALER_CODE',
 'CAL_DATE',
 'CUSTOM_DAY',
 'ACCNT_TYPE_CD',
 'X_CUSTOMER_TYPE',
 'SAP_CODE',
 'ACCOUNT_NAME',
 'ORDER_WID',
 'ROW_WID',
 'X_SUPPLIER_WID',
 'ORDER_TYPE',
 'PROD_WID',
 'X_CITY_CATEGORY',
 'X_FINANCE_FLG',
 'FINANCIER_NAME',
 'X_EXCHANGE_FLG',
 'MODEL',
 'SKU',
 'HMCL_BILLING_TYPE',
 'INVOICE_AMOUNT',
 'CGST',
 'SGST',
 'UTGST',
 'IGST',
 'CESS',
 'INVOICED_SALES',
 'CANCELLED_SALES',
 'RETURNED_SALES',
 'ORDER_NUM',
 'UPGRADE_STATUS',
 'NET_SALES']

In [ ]:
import pandas as pd

# ── Step 1: Split PARENT_DEALER_CODE_MODEL_FAMILY ─────────────────────────────
forecast_df[['DEALER_CODE', 'MODEL_FAMILY_CODE']] = (
    forecast_df['PARENT_DEALER_CODE_MODEL_FAMILY']
    .str.split('<>', expand=True)
)

# Rename forecast NET_SALES immediately to avoid collision
forecast_df = forecast_df.rename(columns={'NET_SALES': 'PREDICTED_SALES_FAMILY'})

# ── Step 2: Define the 3-month lookback window ────────────────────────────────
LOOKBACK_START = '2025-10-01'
LOOKBACK_END   = '2025-12-31'

# ── Step 3: Aggregate ecr_sales to dealer + SKU level for the lookback window ─
ecr_sales['CAL_DATE'] = pd.to_datetime(ecr_sales['CAL_DATE'])

sku_history = (
    ecr_sales[
        (ecr_sales['CAL_DATE'] >= LOOKBACK_START) &
        (ecr_sales['CAL_DATE'] <= LOOKBACK_END)
    ]
    .groupby(['DEALER_CODE', 'SKU'], as_index=False)
    ['NET_SALES']
    .sum()
    .rename(columns={'NET_SALES': 'SKU_SALES_3M'})
)

# ── Step 4: Bring in MODEL_FAMILY_CODE via sku_model_family_mapping ───────────
sku_history = sku_history.merge(
    sku_model_family_mapping[['SKU', 'MODEL_FAMILY_CODE']],
    on='SKU',
    how='left'
)

# ── Step 5: Compute total family sales per dealer in the lookback window ──────
family_totals = (
    sku_history
    .groupby(['DEALER_CODE', 'MODEL_FAMILY_CODE'], as_index=False)
    ['SKU_SALES_3M']
    .sum()
    .rename(columns={'SKU_SALES_3M': 'FAMILY_SALES_3M'})
)

# ── Step 6: Compute SKU contribution % within each dealer-family ──────────────
sku_weights = sku_history.merge(family_totals, on=['DEALER_CODE', 'MODEL_FAMILY_CODE'], how='left')
sku_weights['SKU_WEIGHT'] = sku_weights['SKU_SALES_3M'] / sku_weights['FAMILY_SALES_3M']
sku_weights['NO_HISTORY_FLAG'] = sku_weights['FAMILY_SALES_3M'] == 0

# ── Step 7: Join forecast with SKU weights ────────────────────────────────────
disaggregated = forecast_df.merge(
    sku_weights[['DEALER_CODE', 'MODEL_FAMILY_CODE', 'SKU', 'SKU_WEIGHT', 'NO_HISTORY_FLAG']],
    on=['DEALER_CODE', 'MODEL_FAMILY_CODE'],
    how='left'
)

# ── Step 8: Disaggregate predicted sales ──────────────────────────────────────
disaggregated['PREDICTED_SALES_SKU'] = (
    disaggregated['PREDICTED_SALES_FAMILY'] * disaggregated['SKU_WEIGHT']
).round(0)

disaggregated.loc[disaggregated['NO_HISTORY_FLAG'] == True, 'PREDICTED_SALES_SKU'] = None

# ── Step 9: Final output ──────────────────────────────────────────────────────
output = disaggregated[[
    'MONTH_OF_SALE',
    'PARENT_DEALER_CODE_MODEL_FAMILY',
    'DEALER_CODE',
    'MODEL_FAMILY_CODE',
    'SKU',
    'SKU_WEIGHT',
    'PREDICTED_SALES_FAMILY',
    'PREDICTED_SALES_SKU',
    'NO_HISTORY_FLAG'
]].sort_values(['MONTH_OF_SALE', 'PARENT_DEALER_CODE_MODEL_FAMILY', 'SKU']).reset_index(drop=True)

# ── Sanity check ──────────────────────────────────────────────────────────────
check = (
    output[output['NO_HISTORY_FLAG'] == False]
    .groupby(['MONTH_OF_SALE', 'PARENT_DEALER_CODE_MODEL_FAMILY'])
    .agg(
        ORIGINAL_FORECAST = ('PREDICTED_SALES_FAMILY', 'first'),
        REAGGREGATED_SUM  = ('PREDICTED_SALES_SKU',    'sum')
    )
)
check['DIFF'] = (check['ORIGINAL_FORECAST'] - check['REAGGREGATED_SUM']).abs()
print(f"Max reaggregation diff (rounding): {check['DIFF'].max()}")
print(f"Rows with NO_HISTORY_FLAG        : {output['NO_HISTORY_FLAG'].sum()}")
print(f"Output shape                     : {output.shape}")
output.head(10)